# grid-rbd inline CUDA — compile + run a kernel against the generated header

This is the **tutorial version of [`../cuda/`](../cuda/)**: instead of the
`grid_rbd` Python bindings (the other notebooks), we go one layer down and use
GRiD the way a controls/MPC author would — **generate a `grid.cuh` from a URDF,
write our own CUDA kernel that calls the `grid::` device functions, compile it
with `nvcc` right here in the notebook, run it, and validate the output against
`RBDReference`.**

We drive `inverse_dynamics` (the RNEA — the simplest algorithm) on the
fixed-base KUKA **iiwa14** (7 DoF), mirroring
`examples/cuda/inverse_dynamics_kernel_example.cu`.

**Setup:** a CUDA GPU + **`nvcc` on PATH**, and the repo on `sys.path` so
`URDFParser` / `GRiDCodeGenerator` / `RBDReference` import (run from the repo
root, or `pip install -e python/` is enough for the bindings but here we also
import the codegen packages, so run this notebook from inside the repo tree).
This notebook ends in an `assert` so a green *Run All* validates the *numbers*.

In [ ]:
import os, subprocess, shutil, tempfile
from pathlib import Path
import numpy as np

# Locate the repo root (the dir that holds robot_assets/ + the codegen packages).
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'robot_assets' / 'iiwa14.urdf').exists())
URDF = REPO / 'robot_assets' / 'iiwa14.urdf'
assert URDF.exists(), URDF

NVCC = shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
assert Path(NVCC).exists(), f'nvcc not found at {NVCC}'
print('repo :', REPO)
print('nvcc :', NVCC)

## 1. Generate `grid.cuh` for iiwa14 (inverse_dynamics only)

`GRiDCodeGenerator(...).gen_all_code(algorithm_list=[...])` emits the CUDA
header. Restricting `algorithm_list` to just `inverse_dynamics` keeps the header
small and `nvcc` fast. This is exactly what `examples/codegen/generate_iiwa14.py`
and `examples/cuda/gen_iiwa14_header.py` do, inline.

In [ ]:
import sys
sys.path.insert(0, str(REPO))   # so URDFParser / GRiDCodeGenerator import
from URDFParser import URDFParser
from GRiDCodeGenerator import GRiDCodeGenerator

workdir = Path(tempfile.mkdtemp(prefix='grid_inline_cuda_'))
header = workdir / 'grid.cuh'

robot = URDFParser().parse(str(URDF), floating_base=False)
GRiDCodeGenerator(robot, FILE_NAMESPACE='grid').gen_all_code(
    algorithm_list=['inverse_dynamics'], output_path=str(header))
NJ = robot.get_num_joints()
print(f'wrote {header}  (robot={robot.name}, DOF={NJ}, {header.stat().st_size//1024} KB)')

## 2. Write a small CUDA kernel that calls `grid::inverse_dynamics_device`

The single-block pattern from `examples/cuda/README.md`:

1. `#include "grid.cuh"` — everything lives in namespace `grid::`.
2. Reserve the **dynamic shared memory** the algorithm needs via the emitted
   `grid::INVERSE_DYNAMICS_DEVICE_DYNAMIC_SHARED_MEM_BYTES<T>()` macro
   (`cudaFuncSetAttribute(..., cudaFuncAttributeMaxDynamicSharedMemorySize, ...)`)
   **before** launching — skipping this makes the launch fail and silently zero
   the output (the #1 GRiD-kernel footgun, so we also check `cudaGetLastError`).
3. Call the `_device` auto-scratch wrapper: it carves `s_vaf` / `s_XImats` /
   `s_temp` out of that arena, runs `load_update_XImats_helpers`, then the inner.

We use deterministic inputs identical to `examples/cuda/validate.py`
(`q[i]=0.1*(i+1)`, `qd[i]=0.01*(i+1)`, `qdd[i]=0.02*(i+1)`) so the numpy oracle
and the kernel see the same state.

In [ ]:
kernel_cu = workdir / 'id_kernel.cu'
kernel_cu.write_text(r'''#include <cstdio>
#include <vector>
#include "grid.cuh"   // generated; defines grid::NUM_JOINTS, the kernels, init_*

// One block, one robot. _device is the auto-scratch wrapper: hand it shared-mem
// inputs/outputs only; it allocates s_vaf/s_XImats/s_temp from the dynamic arena
// we reserve at launch.
template <typename T>
__global__ void id_kernel(T *d_c, const T *d_q, const T *d_qd, const T *d_qdd,
                          const grid::robotModel<T> *d_robotModel, const T gravity) {
    const int n = grid::NUM_JOINTS;
    __shared__ T s_q[grid::NUM_JOINTS], s_qd[grid::NUM_JOINTS];
    __shared__ T s_qdd[grid::NUM_JOINTS], s_c[grid::NUM_JOINTS];
    for (int i = threadIdx.x; i < n; i += blockDim.x) {
        s_q[i] = d_q[i]; s_qd[i] = d_qd[i]; s_qdd[i] = d_qdd[i];
    }
    __syncthreads();
    grid::inverse_dynamics_device<T>(
        s_c, s_q, s_qd, s_qdd, d_robotModel, /*d_f_ext=*/nullptr, gravity);
    __syncthreads();
    for (int i = threadIdx.x; i < n; i += blockDim.x) d_c[i] = s_c[i];
}

template <typename T>
static void run() {
    const T gravity = static_cast<T>(-9.81);
    const int n = grid::NUM_JOINTS;
    // One warp is plenty for a 7-DoF arm; never exceed the launch_bounds cap.
    int threads = 32;
    if (threads > grid::MAX_PERF_LEVEL_THREADS) threads = grid::MAX_PERF_LEVEL_THREADS;

    cudaStream_t *streams = grid::init_grid<T>();
    grid::robotModel<T> *d_robotModel = grid::init_robotModel<T>();

    std::vector<T> h_q(n), h_qd(n), h_qdd(n);
    for (int i = 0; i < n; ++i) {
        h_q[i] = (T)(0.1 * (i + 1)); h_qd[i] = (T)(0.01 * (i + 1)); h_qdd[i] = (T)(0.02 * (i + 1));
    }
    T *d_q, *d_qd, *d_qdd, *d_c;
    gpuErrchk(cudaMalloc(&d_q,   n * sizeof(T)));
    gpuErrchk(cudaMalloc(&d_qd,  n * sizeof(T)));
    gpuErrchk(cudaMalloc(&d_qdd, n * sizeof(T)));
    gpuErrchk(cudaMalloc(&d_c,   n * sizeof(T)));
    gpuErrchk(cudaMemcpy(d_q,   h_q.data(),   n * sizeof(T), cudaMemcpyHostToDevice));
    gpuErrchk(cudaMemcpy(d_qd,  h_qd.data(),  n * sizeof(T), cudaMemcpyHostToDevice));
    gpuErrchk(cudaMemcpy(d_qdd, h_qdd.data(), n * sizeof(T), cudaMemcpyHostToDevice));

    // Register the opt-in dynamic shared memory BEFORE launching, then CHECK errors.
    const size_t smem = grid::INVERSE_DYNAMICS_DEVICE_DYNAMIC_SHARED_MEM_BYTES<T>();
    gpuErrchk(cudaFuncSetAttribute(id_kernel<T>,
        cudaFuncAttributeMaxDynamicSharedMemorySize, (int)smem));
    id_kernel<T><<<1, threads, smem>>>(d_c, d_q, d_qd, d_qdd, d_robotModel, gravity);
    gpuErrchk(cudaPeekAtLastError());     // catches a silent launch-config failure
    gpuErrchk(cudaDeviceSynchronize());   // catches in-kernel faults

    std::vector<T> h_c(n);
    gpuErrchk(cudaMemcpy(h_c.data(), d_c, n * sizeof(T), cudaMemcpyDeviceToHost));
    printf("BEGIN inverse_dynamics\n");
    for (int i = 0; i < n; ++i) printf("%.10g%s", (double)h_c[i], i + 1 < n ? " " : "\n");
    printf("END inverse_dynamics\n");

    cudaFree(d_q); cudaFree(d_qd); cudaFree(d_qdd); cudaFree(d_c);
    gpuErrchk(cudaFree(d_robotModel));
    for (int i = 0; i < 3; ++i) gpuErrchk(cudaStreamDestroy(streams[i]));
    free(streams);
}

int main() { run<float>(); return 0; }
''')
print('wrote', kernel_cu, f'({kernel_cu.stat().st_size} bytes)')

## 3. Detect the GPU arch and compile with `nvcc`

We query the device's compute capability so the example builds on any GPU
(`sm_120` on an RTX 5090, `sm_86` on an A6000, ...). The generated header
**vendors GLASS inline**, so `-I <workdir>` (for `grid.cuh`) is the only include
we need.

In [ ]:
# Compute capability of device 0, e.g. '120' for sm_120.
cc = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap',
                     '--format=csv,noheader,nounits'],
                    capture_output=True, text=True)
arch = cc.stdout.split('\n')[0].strip().replace('.', '') if cc.returncode == 0 else '86'
binpath = workdir / 'id_example'
cmd = [NVCC, f'-arch=sm_{arch}', '-std=c++17',
       '-I', str(workdir),   # grid.cuh lives here; it vendors GLASS inline
       str(kernel_cu), '-o', str(binpath)]
print('compiling for sm_%s ...' % arch)
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout); print(r.stderr)
assert r.returncode == 0, 'nvcc failed'
print('compiled ->', binpath)

## 4. Run the kernel and parse its output

In [ ]:
r = subprocess.run([str(binpath)], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
print(r.stdout)

# Pull the labelled BEGIN/END block out of stdout (same protocol as validate.py).
def parse_block(text, label):
    grab, vals = False, []
    for line in text.splitlines():
        if line.strip() == f'BEGIN {label}': grab = True; continue
        if line.strip() == f'END {label}':   break
        if grab and line.strip(): vals += [float(x) for x in line.split()]
    return np.array(vals)

c_cuda = parse_block(r.stdout, 'inverse_dynamics')
print('CUDA inverse_dynamics:', np.round(c_cuda, 4))

## 5. Validate vs `RBDReference` (the smoke-test assertion)

Feed the same deterministic inputs through the numpy oracle and diff. float32
RNEA matches the float64 reference to a tiny relative error.

In [ ]:
from RBDReference import RBDReference
ref = RBDReference(robot)
q   = np.array([0.1  * (i + 1) for i in range(NJ)])
qd  = np.array([0.01 * (i + 1) for i in range(NJ)])
qdd = np.array([0.02 * (i + 1) for i in range(NJ)])

c_ref = ref.inverse_dynamics(q, qd, qdd, GRAVITY=-9.81)
if isinstance(c_ref, tuple): c_ref = c_ref[0]
c_ref = np.asarray(c_ref).ravel()

rel = np.linalg.norm(c_cuda - c_ref) / max(np.linalg.norm(c_ref), 1e-9)
print('RBDReference       :', np.round(c_ref, 4))
print(f'relative error     : {rel:.3e}  (float32 RNEA, tol 1e-4)')
assert rel < 1e-4, rel
print('OK: inline-compiled CUDA inverse_dynamics matches RBDReference')

In [ ]:
# Tidy up the scratch dir.
shutil.rmtree(workdir, ignore_errors=True)

That's the whole codegen → compile → run → validate loop, inline. For the
full hand-written-kernel walkthrough (the `_inner` full-control surface, the
`_host` surface for heavy second-order kernels, and batched one-block-per-
timestep launches) see [`../cuda/`](../cuda/) and its
[`README.md`](../cuda/README.md).